# 05 — Final Speaker ↔ Transcript Alignment

Notebook นี้ใช้ข้อมูล final เท่านั้น และออกแบบมาแทน Notebook Speaker Alignment ตัวเก่า

Input:
- `dataset/mouth_crops_final/video001/mouth_metadata.csv`
- `dataset/transcripts/video001/transcript_with_frames.csv`

Output:
- `dataset/final_alignment/video001/final_speaker_transcript_alignment.csv`
- `dataset/final_alignment/video001/final_speaker_transcript_auto.csv`
- `dataset/final_alignment/video001/final_speaker_transcript_review.csv`
- `dataset/final_alignment/video001/segment_speaker_activity.csv`
- `dataset/final_alignment/video001/mouth_motion.csv`

หลักการ:
1. ใช้ mouth crops ของ `speaker_id 0/1`
2. คำนวณ lip motion ระหว่างเฟรม
3. สรุป activity ภายใน transcript segment
4. เปรียบเทียบ Speaker 0 กับ Speaker 1
5. ช่วงที่มั่นใจสูง → `AUTO`
6. ช่วงที่สูสี / ข้อมูลไม่พอ → `REVIEW` หรือ `NO_DATA`

## Cell 1 — Path / Config

In [1]:
from pathlib import Path

PROJECT_ROOT = Path(r"D:\Face_detection")
VIDEO_ID = "video001"

MOUTH_METADATA_CSV = (
    PROJECT_ROOT / "dataset" / "mouth_crops_final"
    / VIDEO_ID / "mouth_metadata.csv"
)

TRANSCRIPT_CSV = (
    PROJECT_ROOT / "dataset" / "transcripts"
    / VIDEO_ID / "transcript_with_frames.csv"
)

OUTPUT_ROOT = (
    PROJECT_ROOT / "dataset" / "final_alignment" / VIDEO_ID
)
OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)

print("MOUTH_METADATA_CSV:", MOUTH_METADATA_CSV)
print("TRANSCRIPT_CSV    :", TRANSCRIPT_CSV)
print("OUTPUT_ROOT       :", OUTPUT_ROOT)

if not MOUTH_METADATA_CSV.exists():
    raise FileNotFoundError(MOUTH_METADATA_CSV)

if not TRANSCRIPT_CSV.exists():
    raise FileNotFoundError(TRANSCRIPT_CSV)

MOUTH_METADATA_CSV: D:\Face_detection\dataset\mouth_crops_final\video001\mouth_metadata.csv
TRANSCRIPT_CSV    : D:\Face_detection\dataset\transcripts\video001\transcript_with_frames.csv
OUTPUT_ROOT       : D:\Face_detection\dataset\final_alignment\video001


## Cell 2 — Load Data

In [2]:
import pandas as pd
import numpy as np

mouth_df = pd.read_csv(MOUTH_METADATA_CSV)
transcript_df = pd.read_csv(TRANSCRIPT_CSV)

if mouth_df["accepted"].dtype == object:
    mouth_df["accepted"] = (
        mouth_df["accepted"].astype(str).str.lower()
        .isin(["true", "1", "yes"])
    )

mouth_df = mouth_df[mouth_df["accepted"] == True].copy()
mouth_df["frame_index"] = mouth_df["frame_index"].astype(int)
mouth_df["speaker_id"] = mouth_df["speaker_id"].astype(int)

transcript_df["start_frame"] = transcript_df["start_frame"].astype(int)
transcript_df["end_frame"] = transcript_df["end_frame"].astype(int)

print("Accepted mouth rows :", len(mouth_df))
print("Speaker IDs         :", sorted(mouth_df["speaker_id"].unique().tolist()))
print("Transcript segments :", len(transcript_df))

display(mouth_df.head())
display(transcript_df.head())

Accepted mouth rows : 253684
Speaker IDs         : [0, 1]
Transcript segments : 1293


,video_id,frame_index,timestamp_seconds,speaker_id,track_id,det_score,mouth_file,blur_score,brightness,accepted
0,video001,0,0.000000,0,0,0.833094,dataset\mouth_crops_final\video001\speaker_000...,30.776,140.680,True
1,video001,0,0.000000,1,1,0.824791,dataset\mouth_crops_final\video001\speaker_001...,78.089,159.312,True
2,video001,1,0.033367,0,0,0.834487,dataset\mouth_crops_final\video001\speaker_000...,23.251,140.616,True
3,video001,1,0.033367,1,1,0.826267,dataset\mouth_crops_final\video001\speaker_001...,68.047,160.058,True
4,video001,2,0.066733,0,0,0.834542,dataset\mouth_crops_final\video001\speaker_000...,23.032,140.117,True


,video_id,segment_id,start_time,end_time,start_frame,end_frame,frame_count,text
0,video001,0,0.42,2.20,12,66,55,และนี่คือ TiaMai Podcast
1,video001,1,2.48,4.44,74,134,61,Podcast ที่จะชวนคุณมาวิคราบไปด้วยกัน
2,video001,2,4.44,6.50,133,195,63,ว่าถ้ามันเกิดขึ้นกับคุณ คุณจะทำยังไง
3,video001,3,6.50,7.94,194,238,45,ใช่ค่ะ และทั้งหมดนะครับ
4,video001,4,7.94,9.74,237,292,56,เป็นเพลงแค่ความคิดเห็นของเราสองคน


## Cell 3 — Coverage QA

In [3]:
max_mouth_frame = int(mouth_df["frame_index"].max())
max_transcript_frame = int(transcript_df["end_frame"].max())

coverage = (
    max_mouth_frame / max_transcript_frame
    if max_transcript_frame > 0 else 0.0
)

print("Max mouth frame     :", max_mouth_frame)
print("Max transcript frame:", max_transcript_frame)
print("Approx coverage     :", f"{coverage:.2%}")

if coverage < 0.90:
    print("\n[WARNING] mouth data ยังครอบคลุม transcript ไม่ถึง 90%")
else:
    print("\nCoverage OK")

Max mouth frame     : 178333
Max transcript frame: 178258
Approx coverage     : 100.04%

Coverage OK


## Cell 4 — Mouth Motion Functions

In [4]:
import cv2
from pathlib import Path

def resolve_mouth_path(relative_path):
    if not isinstance(relative_path, str):
        return None
    relative_path = relative_path.strip()
    if not relative_path:
        return None

    p = Path(relative_path)
    return p if p.is_absolute() else PROJECT_ROOT / p


def read_mouth_gray(relative_path):
    p = resolve_mouth_path(relative_path)

    if p is None or not p.exists():
        return None

    img = cv2.imread(str(p), cv2.IMREAD_GRAYSCALE)

    if img is None:
        return None

    if img.shape != (96, 96):
        img = cv2.resize(
            img, (96, 96),
            interpolation=cv2.INTER_AREA
        )

    return img


def frame_difference(img_a, img_b):
    if img_a is None or img_b is None:
        return np.nan

    return float(
        cv2.absdiff(img_a, img_b).mean() / 255.0
    )

print("Motion functions ready")

Motion functions ready


## Cell 5 — Build Mouth Motion Table

In [5]:
from tqdm.auto import tqdm

motion_rows = []

for speaker_id, group in mouth_df.groupby("speaker_id"):

    group = group.sort_values("frame_index").reset_index(drop=True)

    prev_img = None
    prev_frame = None

    for _, row in tqdm(
        group.iterrows(),
        total=len(group),
        desc=f"speaker {speaker_id}"
    ):
        current_frame = int(row["frame_index"])
        current_img = read_mouth_gray(row["mouth_file"])

        if (
            prev_img is not None
            and prev_frame is not None
            and current_frame - prev_frame <= 2
        ):
            motion = frame_difference(prev_img, current_img)
        else:
            motion = np.nan

        motion_rows.append({
            "speaker_id": int(speaker_id),
            "frame_index": current_frame,
            "motion": motion,
            "blur_score": row.get("blur_score", np.nan),
            "brightness": row.get("brightness", np.nan),
            "det_score": row.get("det_score", np.nan),
        })

        prev_img = current_img
        prev_frame = current_frame

motion_df = pd.DataFrame(motion_rows)

print("Motion rows:", len(motion_df))
display(motion_df.head(20))

c:\Users\ASUS FA507NUR\AppData\Local\Programs\Python\Python312\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
speaker 1: 100%|██████████| 136417/136417 [29:24<00:00, 77.33it/s]


Motion rows: 253684


,speaker_id,frame_index,motion,blur_score,brightness,det_score
0,0,0,NaN,30.776,140.680,0.833094
1,0,1,0.009124,23.251,140.616,0.834487
2,0,2,0.011507,23.032,140.117,0.834542
3,0,3,0.005044,22.551,139.968,0.835783
4,0,4,0.014334,21.358,139.707,0.832522
5,0,5,0.013756,20.253,139.690,0.832506
6,0,6,0.009517,19.632,139.726,0.832968
7,0,7,0.010345,19.993,139.655,0.832316
8,0,8,0.011656,20.454,139.985,0.832112
9,0,9,0.011025,20.948,139.558,0.835347


## Cell 6 — Smooth Motion

In [6]:
motion_df = motion_df.sort_values(
    ["speaker_id", "frame_index"]
).reset_index(drop=True)

motion_df["motion_smooth"] = (
    motion_df
    .groupby("speaker_id")["motion"]
    .transform(
        lambda s: s.rolling(
            window=5,
            center=True,
            min_periods=1
        ).median()
    )
)

display(motion_df.head(20))

,speaker_id,frame_index,motion,blur_score,brightness,det_score,motion_smooth
0,0,0,NaN,30.776,140.680,0.833094,0.010315
1,0,1,0.009124,23.251,140.616,0.834487,0.009124
2,0,2,0.011507,23.032,140.117,0.834542,0.010315
3,0,3,0.005044,22.551,139.968,0.835783,0.011507
4,0,4,0.014334,21.358,139.707,0.832522,0.011507
5,0,5,0.013756,20.253,139.690,0.832506,0.010345
6,0,6,0.009517,19.632,139.726,0.832968,0.011656
7,0,7,0.010345,19.993,139.655,0.832316,0.011025
8,0,8,0.011656,20.454,139.985,0.832112,0.010345
9,0,9,0.011025,20.948,139.558,0.835347,0.010345


## Cell 7 — Activity per Transcript Segment

In [7]:
segment_activity_rows = []

for _, seg in tqdm(
    transcript_df.iterrows(),
    total=len(transcript_df),
    desc="transcript segments"
):
    segment_id = int(seg["segment_id"])
    start_frame = int(seg["start_frame"])
    end_frame = int(seg["end_frame"])
    expected_frames = max(1, end_frame - start_frame + 1)

    matched = motion_df[
        (motion_df["frame_index"] >= start_frame)
        & (motion_df["frame_index"] <= end_frame)
    ]

    for speaker_id in [0, 1]:
        speaker_data = matched[
            matched["speaker_id"] == speaker_id
        ]

        valid_motion = speaker_data["motion_smooth"].dropna()
        observed_frames = int(
            speaker_data["frame_index"].nunique()
        )

        segment_activity_rows.append({
            "segment_id": segment_id,
            "speaker_id": speaker_id,
            "mean_motion": (
                float(valid_motion.mean())
                if len(valid_motion) else np.nan
            ),
            "median_motion": (
                float(valid_motion.median())
                if len(valid_motion) else np.nan
            ),
            "peak_motion": (
                float(valid_motion.quantile(0.90))
                if len(valid_motion) else np.nan
            ),
            "observed_frames": observed_frames,
            "coverage": min(
                1.0, observed_frames / expected_frames
            ),
        })

activity_df = pd.DataFrame(segment_activity_rows)

display(activity_df.head(20))

transcript segments: 100%|██████████| 1293/1293 [00:04<00:00, 273.09it/s]


,segment_id,speaker_id,mean_motion,median_motion,peak_motion,observed_frames,coverage
0,0,0,0.014367,0.014361,0.021145,55,1.000000
1,0,1,0.024163,0.026042,0.032665,55,1.000000
2,1,0,0.012659,0.013017,0.018233,61,1.000000
3,1,1,0.024275,0.024507,0.033525,61,1.000000
4,2,0,0.015166,0.013374,0.021839,63,1.000000
5,2,1,0.020683,0.019000,0.030441,63,1.000000
6,3,0,0.020485,0.019627,0.029856,45,1.000000
7,3,1,0.013826,0.012555,0.020602,45,1.000000
8,4,0,0.022402,0.023121,0.028034,56,1.000000
9,4,1,0.009973,0.008917,0.014248,56,1.000000


## Cell 8 — Speaker Activity Score

In [8]:
activity_df["mean_motion"] = activity_df["mean_motion"].fillna(0.0)
activity_df["peak_motion"] = activity_df["peak_motion"].fillna(0.0)
activity_df["coverage"] = activity_df["coverage"].fillna(0.0)

activity_df["speaker_score"] = (
    0.55 * activity_df["mean_motion"]
    + 0.30 * activity_df["peak_motion"]
    + 0.15 * activity_df["coverage"]
)

display(
    activity_df.sort_values(
        ["segment_id", "speaker_score"],
        ascending=[True, False]
    ).head(30)
)

,segment_id,speaker_id,mean_motion,median_motion,peak_motion,observed_frames,coverage,speaker_score
1,0,1,0.024163,0.026042,0.032665,55,1.000000,0.173089
0,0,0,0.014367,0.014361,0.021145,55,1.000000,0.164246
3,1,1,0.024275,0.024507,0.033525,61,1.000000,0.173409
2,1,0,0.012659,0.013017,0.018233,61,1.000000,0.162432
5,2,1,0.020683,0.019000,0.030441,63,1.000000,0.170508
4,2,0,0.015166,0.013374,0.021839,63,1.000000,0.164893
6,3,0,0.020485,0.019627,0.029856,45,1.000000,0.170224
7,3,1,0.013826,0.012555,0.020602,45,1.000000,0.163785
8,4,0,0.022402,0.023121,0.028034,56,1.000000,0.170731
9,4,1,0.009973,0.008917,0.014248,56,1.000000,0.159760


## Cell 9 — Final Speaker Assignment

In [9]:
MIN_COVERAGE = 0.50
MIN_MEAN_MOTION = 0.008
MIN_RELATIVE_MARGIN = 0.18

alignment_rows = []

for _, seg in transcript_df.iterrows():

    segment_id = int(seg["segment_id"])

    candidates = activity_df[
        activity_df["segment_id"] == segment_id
    ].sort_values("speaker_score", ascending=False)

    if len(candidates) == 0:
        continue

    top = candidates.iloc[0]
    second = candidates.iloc[1] if len(candidates) > 1 else None

    top_score = float(top["speaker_score"])
    second_score = (
        float(second["speaker_score"])
        if second is not None else 0.0
    )

    margin = (
        (top_score - second_score)
        / max(top_score, 1e-8)
    )

    top_coverage = float(top["coverage"])
    top_mean_motion = float(top["mean_motion"])

    if top_coverage < MIN_COVERAGE:
        status = "NO_DATA"
    elif top_mean_motion < MIN_MEAN_MOTION:
        status = "REVIEW"
    elif margin < MIN_RELATIVE_MARGIN:
        status = "REVIEW"
    else:
        status = "AUTO"

    alignment_rows.append({
        "video_id": VIDEO_ID,
        "segment_id": segment_id,
        "start_time": float(seg["start_time"]),
        "end_time": float(seg["end_time"]),
        "start_frame": int(seg["start_frame"]),
        "end_frame": int(seg["end_frame"]),
        "text": str(seg["text"]),
        "speaker_id": (
            int(top["speaker_id"])
            if status != "NO_DATA"
            else np.nan
        ),
        "speaker_score": round(top_score, 6),
        "second_score": round(second_score, 6),
        "relative_margin": round(float(margin), 6),
        "coverage": round(top_coverage, 6),
        "mean_motion": round(top_mean_motion, 6),
        "status": status,
    })

alignment_df = pd.DataFrame(alignment_rows)

display(alignment_df.head(30))

,video_id,segment_id,start_time,end_time,start_frame,end_frame,text,speaker_id,speaker_score,second_score,relative_margin,coverage,mean_motion,status
0,video001,0,0.42,2.20,12,66,และนี่คือ TiaMai Podcast,1.0,0.173089,0.164246,0.051094,1.000000,0.024163,REVIEW
1,video001,1,2.48,4.44,74,134,Podcast ที่จะชวนคุณมาวิคราบไปด้วยกัน,1.0,0.173409,0.162432,0.063298,1.000000,0.024275,REVIEW
2,video001,2,4.44,6.50,133,195,ว่าถ้ามันเกิดขึ้นกับคุณ คุณจะทำยังไง,1.0,0.170508,0.164893,0.032928,1.000000,0.020683,REVIEW
3,video001,3,6.50,7.94,194,238,ใช่ค่ะ และทั้งหมดนะครับ,0.0,0.170224,0.163785,0.037824,1.000000,0.020485,REVIEW
4,video001,4,7.94,9.74,237,292,เป็นเพลงแค่ความคิดเห็นของเราสองคน,0.0,0.170731,0.159760,0.064261,1.000000,0.022402,REVIEW
5,video001,5,9.74,11.90,291,357,ซึ่งเป็นสองคนที่ธรรมดาๆเท่านั้นเนอะ,0.0,0.172732,0.161608,0.064402,1.000000,0.023347,REVIEW
6,video001,6,11.90,15.20,356,456,ถ้าใครรู้สึกยังไงก็สามารถแสดงความคิดเห็นกันได้...,0.0,0.172106,0.161929,0.059129,1.000000,0.021126,REVIEW
7,video001,7,15.20,16.04,455,481,ใช่ค่ะ,1.0,0.164050,0.162536,0.009226,1.000000,0.015504,REVIEW
8,video001,8,20.26,22.40,607,672,และวันนี้อย่างที่ทุกคนเห็นนะคะ,0.0,0.110586,0.101878,0.078746,0.575758,0.024201,REVIEW
9,video001,9,22.40,50.76,671,1522,นี่ๆ,0.0,0.158949,0.132203,0.168266,0.927230,0.019735,REVIEW


## Cell 10 — Summary

In [10]:
print("Total segments:", len(alignment_df))
print()
print(alignment_df["status"].value_counts(dropna=False))

print("\nAUTO speaker counts:")
print(
    alignment_df[
        alignment_df["status"] == "AUTO"
    ]["speaker_id"].value_counts()
)

auto_rate = (
    (alignment_df["status"] == "AUTO").mean()
)

print(
    "\nAUTO segment rate:",
    f"{auto_rate:.2%}"
)

Total segments: 1293

status
AUTO       771
REVIEW     506
NO_DATA     16
Name: count, dtype: int64

AUTO speaker counts:
speaker_id
1.0    463
0.0    308
Name: count, dtype: int64

AUTO segment rate: 59.63%


## Cell 11 — Export Final Alignment

In [11]:
ALL_CSV = OUTPUT_ROOT / "final_speaker_transcript_alignment.csv"
AUTO_CSV = OUTPUT_ROOT / "final_speaker_transcript_auto.csv"
REVIEW_CSV = OUTPUT_ROOT / "final_speaker_transcript_review.csv"
ACTIVITY_CSV = OUTPUT_ROOT / "segment_speaker_activity.csv"
MOTION_CSV = OUTPUT_ROOT / "mouth_motion.csv"

alignment_df.to_csv(
    ALL_CSV,
    index=False,
    encoding="utf-8-sig"
)

alignment_df[
    alignment_df["status"] == "AUTO"
].to_csv(
    AUTO_CSV,
    index=False,
    encoding="utf-8-sig"
)

alignment_df[
    alignment_df["status"] != "AUTO"
].to_csv(
    REVIEW_CSV,
    index=False,
    encoding="utf-8-sig"
)

activity_df.to_csv(
    ACTIVITY_CSV,
    index=False,
    encoding="utf-8-sig"
)

motion_df.to_csv(
    MOTION_CSV,
    index=False,
    encoding="utf-8-sig"
)

print("Saved:")
print(" -", ALL_CSV)
print(" -", AUTO_CSV)
print(" -", REVIEW_CSV)
print(" -", ACTIVITY_CSV)
print(" -", MOTION_CSV)

Saved:
 - D:\Face_detection\dataset\final_alignment\video001\final_speaker_transcript_alignment.csv
 - D:\Face_detection\dataset\final_alignment\video001\final_speaker_transcript_auto.csv
 - D:\Face_detection\dataset\final_alignment\video001\final_speaker_transcript_review.csv
 - D:\Face_detection\dataset\final_alignment\video001\segment_speaker_activity.csv
 - D:\Face_detection\dataset\final_alignment\video001\mouth_motion.csv


## Cell 12 — QA: REVIEW

In [12]:
review_df = alignment_df[
    alignment_df["status"] != "AUTO"
].copy()

display(
    review_df[
        [
            "segment_id",
            "start_time",
            "end_time",
            "text",
            "speaker_id",
            "speaker_score",
            "second_score",
            "relative_margin",
            "coverage",
            "mean_motion",
            "status",
        ]
    ].head(100)
)

,segment_id,start_time,end_time,text,speaker_id,speaker_score,second_score,relative_margin,coverage,mean_motion,status
0,0,0.42,2.20,และนี่คือ TiaMai Podcast,1.0,0.173089,0.164246,0.051094,1.000000,0.024163,REVIEW
1,1,2.48,4.44,Podcast ที่จะชวนคุณมาวิคราบไปด้วยกัน,1.0,0.173409,0.162432,0.063298,1.000000,0.024275,REVIEW
2,2,4.44,6.50,ว่าถ้ามันเกิดขึ้นกับคุณ คุณจะทำยังไง,1.0,0.170508,0.164893,0.032928,1.000000,0.020683,REVIEW
3,3,6.50,7.94,ใช่ค่ะ และทั้งหมดนะครับ,0.0,0.170224,0.163785,0.037824,1.000000,0.020485,REVIEW
4,4,7.94,9.74,เป็นเพลงแค่ความคิดเห็นของเราสองคน,0.0,0.170731,0.159760,0.064261,1.000000,0.022402,REVIEW
...,...,...,...,...,...,...,...,...,...,...,...
221,221,751.09,754.37,โอเคค่ะ,1.0,0.165700,0.148236,0.105393,1.000000,0.016945,REVIEW
224,224,761.12,763.28,เราผิดไหมที่ลืมเอา Power Bank มา,NaN,0.000000,0.000000,0.000000,0.000000,0.000000,NO_DATA
225,225,763.28,765.86,จนโดนแฟนด่าว่าไม่มีสมองนึกคิด,NaN,0.027292,0.020439,0.251121,0.063291,0.019967,NO_DATA
226,226,765.86,769.18,อุ๊ย อันนี้เห็นหัวเรื่อง แต่ยังไม่ได้เข้าไปอ่าน,0.0,0.169380,0.164486,0.028893,1.000000,0.018918,REVIEW


## Cell 13 — QA: AUTO Samples

In [13]:
auto_df = alignment_df[
    alignment_df["status"] == "AUTO"
].copy()

if len(auto_df) > 0:
    sample_auto = auto_df.sample(
        n=min(30, len(auto_df)),
        random_state=42
    ).sort_values("start_time")

    display(
        sample_auto[
            [
                "segment_id",
                "start_time",
                "end_time",
                "text",
                "speaker_id",
                "relative_margin",
                "coverage",
                "mean_motion",
            ]
        ]
    )
else:
    print("ไม่มี AUTO segment")

,segment_id,start_time,end_time,text,speaker_id,relative_margin,coverage,mean_motion
111,111,387.88,391.08,เขาก็กลับพิมมาด้วยข้อความหุ้นๆ ใส่เธออีกครั้ง,0.0,1.000000,1.000000,0.015355
122,122,421.98,425.02,เพราะการพยายามคุยหรือเปิดโอเคลียร์กับเขานี่,0.0,1.000000,1.000000,0.017990
253,253,859.52,861.36,แต่สิ่งที่ทำให้รู้สึกแห้มาก ๆ,1.0,1.000000,1.000000,0.019808
348,348,1282.10,1284.58,แต่นึงจากวันนั้นเป็นงานเลี้ยงวันเกิดของเขา,0.0,1.000000,1.000000,0.016360
399,399,1569.92,1569.94,ค้าต้อง는데 เอาของแม็กไปเจอเธอ,0.0,1.000000,1.000000,0.022214
403,403,1575.04,1577.90,ก็พยายามเปิดบรสดธนาคุยกับเธอ,0.0,1.000000,1.000000,0.017628
404,404,1577.90,1580.44,โดยเขาพยายามแก้ตัวแทนแม่ว่า,0.0,1.000000,1.000000,0.020015
408,408,1585.26,1591.04,พร้อมกับบอกให้เธอเจียนเย็น ๆ แล้วก็บอกเธอว่าเธ...,0.0,1.000000,1.000000,0.015658
501,501,2149.84,2153.22,ฉันคงทำแบบนั้นไม่ได้ค่ะ แต่แฟนตอบกลับไปว่า,1.0,1.000000,1.000000,0.018466
510,510,2178.94,2182.68,และเมื่อเธอสังเกียดเห็นว่าสิรรคาทั้งหมดที่ต่อแ...,1.0,1.000000,1.000000,0.019671


# ขั้นต่อไป — 06 Final Labeled Lip Clip Builder

เมื่อ alignment ผ่าน ให้ใช้:
- `final_speaker_transcript_auto.csv`
- `mouth_metadata.csv`

เพื่อสร้าง labeled clips จริง แล้วค่อยทำ manifest / train / val / test